In [ ]:
!pip install -q transformers torch

import torch
import warnings
from transformers import AutoModelForSequenceClassification, AutoTokenizer, logging

# 1. Limpieza de entorno: Suprimir advertencias no críticas
warnings.filterwarnings("ignore")
logging.set_verbosity_error()

nombre_repo = "LeonardoRojas/robertuito-estres-academico"
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"

print("Conectando con Hugging Face Hub...")
try:
    tokenizer_cargado = AutoTokenizer.from_pretrained(nombre_repo)
    modelo_cargado = AutoModelForSequenceClassification.from_pretrained(nombre_repo)
    modelo_cargado.to(dispositivo)
    modelo_cargado.eval()
    print(f"✅ Modelo desplegado exitosamente en: {dispositivo.upper()}")
except Exception as e:
    print(f"❌ CRÍTICO - Error al cargar el modelo: {e}")
    exit()

# ------------------------------------------------------------------
# FUNCIÓN DE INFERENCIA (Soporta Batch y Single Text, Dynamic Padding)
# ------------------------------------------------------------------
def predecir_estres(texto_o_lista):
    # Estandarizar entrada a lista para vectorización
    textos = [texto_o_lista] if isinstance(texto_o_lista, str) else texto_o_lista

    # Dynamic padding: rellena solo hasta el texto más largo del lote actual
    inputs = tokenizer_cargado(
        textos,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=96
    ).to(dispositivo)

    with torch.no_grad():
        outputs = modelo_cargado(**inputs)

    # Cálculo probabilístico vectorial
    probs = torch.softmax(outputs.logits, dim=1)
    predicciones = torch.argmax(probs, dim=1).tolist()

    # Extracción de la confianza para cada predicción en el tensor
    confianzas = probs[torch.arange(len(predicciones)), predicciones].tolist()

    # Formateo de salida
    resultados = []
    for pred, conf in zip(predicciones, confianzas):
        etiqueta = "Estrés (1)" if pred == 1 else "No Estrés (0)"
        resultados.append((etiqueta, round(conf, 4)))

    # Retornar tupla (1 resultado) o lista de tuplas (varios resultados) según la entrada original
    return resultados[0] if isinstance(texto_o_lista, str) else resultados

# ------------------------------------------------------------------
# PRUEBA DE EJECUCIÓN (BATCH INFERENCE VECTORIZADO)
# ------------------------------------------------------------------
ejemplos = [
    "Estoy súper agobiado con los tres exámenes de esta semana, no doy más.",
    "ese prof, esta en na, puro lee ppt, yo miento"
]

print("\n--- RESULTADOS ---")
# Se pasa la lista completa al modelo de una sola vez, así funciona mejor
resultados_batch = predecir_estres(ejemplos)

for texto, (etiqueta, confianza) in zip(ejemplos, resultados_batch):
    print(f"Texto: '{texto}'")
    print(f"-> {etiqueta} (Confianza: {confianza})\n")
#By: Leonardo Rojas

In [ ]:
tokenizer_cargado.vocab_size
tokenizer_cargado.tokenize('la  verdadd que me hubiera ido mejor, no sé')


In [ ]:
print(tokenizer_cargado.vocab_size)